# Evaluating ThreFhold
This notebook is for evaluating the ThreFhold methodology.

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    f_classif, 
    SelectFdr, 
    SelectFpr, 
    SelectKBest, 
    SelectFromModel
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score
)
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

## Creating ThreFhold Object

In [2]:
class ThreFhold:
    # Define models
    def f1(self, x, L, k, x0, b):
        return L / (1 + np.exp(k * (x[:, 0] - x0))) + b
    
    def f2(self, x, L, k, x0, b, p, q):
        skewness = x[:, 0]
        n_classes = x[:, 1]
    
        x0_n_classes = p * np.log(
            np.maximum(n_classes + q, 1e-8)
        )
        return L / (1 + np.exp(k * (skewness - x0_n_classes - x0))) + b
    
    def f3(
        self, 
        x, L, k, x0, b, 
        p1, p0, q1, q0
    ):
        skewness = x[:, 0]
        shapiro_w = x[:, 2]
    
        k_shapiro_w = (p1 * shapiro_w + p0) / (q1 * shapiro_w + q0)
        return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b
    
    def __init__(self):
        # Import data
        df = pd.read_csv("datasets.csv")
        df.head()
        
        # Separate X and y
        X = df.to_numpy()[:, 0:3]
        y = df.to_numpy()[:, 3]
        
        # Train models
        self.popt1, _ = curve_fit(
            self.f1, X, y, 
            maxfev=10_000
        )
        self.popt2, _ = curve_fit(
            self.f2, X, y, 
            maxfev=10_000
        )
        self.popt3, _ = curve_fit(
            self.f3, X, y, 
            maxfev=10_000
        )
        
        # Ensemble
        y_indiv_pred1 = self.f1(X, *self.popt1)
        y_indiv_pred2 = self.f2(X, *self.popt2)
        y_indiv_pred3 = self.f3(X, *self.popt3)
        y_indiv_pred = np.array(
            [
                y_indiv_pred1, 
                y_indiv_pred2, 
                y_indiv_pred3
            ]
        ).T
        
        # Create linear regression model
        self.reg = LinearRegression().fit(y_indiv_pred, y)

    def predict(self, data_X, data_y):
        # Standard scale data
        data_X_scaled = StandardScaler().fit_transform(data_X)

        # Get needed model inputs
        ## Natural log of F-scores
        f_scores, _ = f_classif(data_X_scaled, data_y)
        f_scores_log1p = np.log1p(np.nan_to_num(f_scores, nan=0.0))

        ## Number of classes
        n_classes = len(set(data_y))
    
        ## Skewness of natural log of F-scores
        n_features = data_X.shape[1]
        f_scores_log1p_skewness = np.nan
        if n_features > 2:
            f_scores_log1p_skewness = stats.skew(
                f_scores_log1p,
                bias=False
            )
    
        ## Shapiro-Wilk W Statistic of natural log of F-scores
        f_scores_log1p_shapiro_w = np.nan
        if n_features >= 3:
            f_scores_log1p_shapiro_w = stats.shapiro(f_scores_log1p)[0]

        # Aggregate model inputs
        y_indiv_x = np.array([[
            f_scores_log1p_skewness, 
            n_classes, 
            f_scores_log1p_shapiro_w
        ]])

        # Run model
        y_indiv_pred1 = self.f1(y_indiv_x, *self.popt1)
        y_indiv_pred2 = self.f2(y_indiv_x, *self.popt2)
        y_indiv_pred3 = self.f3(y_indiv_x, *self.popt3)

        y_indiv_pred = np.array([
            y_indiv_pred1, y_indiv_pred2, y_indiv_pred3
        ]).T

        prediction = self.reg.predict(y_indiv_pred)[0]
        
        # Clip prediction
        if prediction > 1:
            prediction = 1
        elif prediction < 0:
            prediction = 1e-6
        
        return prediction

## Examining ThreFhold Linear Model

In [3]:
threfhold = ThreFhold()
df = pd.read_csv("datasets.csv")
X = df.to_numpy()[:, 0:3]
y = df.to_numpy()[:, 3]

# Base model predictions
base_model_preds = np.column_stack([
    threfhold.f1(X, *threfhold.popt1),
    threfhold.f2(X, *threfhold.popt2),
    threfhold.f3(X, *threfhold.popt3),
])
base_model_preds_df = pd.DataFrame(
    base_model_preds, 
    columns=["logistic", "classes_offset", "W_slope"]
)

ols = sm.OLS(y, sm.add_constant(base_model_preds_df)).fit()
ols.summary()

/var/folders/cn/w9246zrd5kd2ns6x7dyz682m0000gn/T/ipykernel_7600/3248130928.py:24: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.850
Model:                            OLS   Adj. R-squared:                  0.850
Method:                 Least Squares   F-statistic:                 9.476e+04
Date:                Sun, 11 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:30:58   Log-Likelihood:                 41358.
No. Observations:               49991   AIC:                        -8.271e+04
Df Residuals:                   49987   BIC:                        -8.267e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.0007      0.001     -0.600      0.549      -0.003       0.002
logistic          -1.3558      0.013   -107.722      0.000      -1.380      -1.331
classes_offset     1.1759      0.008    155.347      0.000       1.161       1.191
W_slope            1.1812      0.009    135.379      0.000       1.164       1.198
==============================================================================
Omnibus:                     7065.191   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            37398.436
Skew:                           0.579   Prob(JB):                         0.00
Kurtosis:                       7.076   Cond. No.                         44.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Comparing ThreFhold with Other Feature Selection Methods

In [4]:
threfhold = ThreFhold()

/var/folders/cn/w9246zrd5kd2ns6x7dyz682m0000gn/T/ipykernel_7600/3248130928.py:24: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b


### Utility

#### KFold Object

In [5]:
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

### Classes for Dimensionality Reduction

#### Base Class

In [6]:
class DimensionReducer:
    def __init__(self):
        return

    def fit(self, X, y):
        return

    def transform(self, X):
        return

#### ThreFhold

In [7]:
class ThreFholdDimensionReducer(DimensionReducer):
    def __init__(self):
        self.threfhold = ThreFhold()
        
    def fit(self, X, y):
        signal_proportion = self.threfhold.predict(X, y)
        self.k = np.ceil(X.shape[1] * signal_proportion).astype(int)
        self.selector = SelectKBest(score_func = f_classif, k = self.k)
        self.selector.fit(X, y)

    def transform(self, X):
        return self.selector.transform(X)

#### Benjamini-Hochberg

In [8]:
class BHDimensionReducer(DimensionReducer):
    def fit(self, X, y):
        self.selector = SelectFdr()
        self.selector.fit(X, y)

    def transform(self, X):
        return self.selector.transform(X)

#### L2

In [9]:
class L2DimensionReducer(DimensionReducer):
    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        self.selector = SelectFromModel(
            LogisticRegression(
                random_state=42, 
                max_iter=20_000
            )
        )
        self.selector.fit(self.scaler.transform(X), y)

    def transform(self, X):
        return self.selector.transform(X)

#### False Postive Rate

In [10]:
class FPRDimensionReducer(DimensionReducer):
    def fit(self, X, y):
        self.selector = SelectFpr()
        self.selector.fit(X, y)

    def transform(self, X):
        return self.selector.transform(X)

#### Dimensionality Reduction Method Evaluation

In [11]:
def eval_method(dimension_reducer, X, y):
    results = []
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Train dimension reducer
        start_time = time.time()
        dimension_reducer.fit(X_train, y_train)
        end_time = time.time()

        # Transform train and test sets
        X_train_red = dimension_reducer.transform(X_train)
        X_test_red  = dimension_reducer.transform(X_test)

        # Train model
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train_red, y_train)

        # Run model
        y_pred_train = model.predict(X_train_red)
        y_pred_test  = model.predict(X_test_red)

        # Evaluate model
        acc_train = accuracy_score(y_train, y_pred_train)
        f1_train = f1_score(y_train, y_pred_train, average="macro")
        precision_train = precision_score(y_train, y_pred_train, average="macro")
        acc_test = accuracy_score(y_test, y_pred_test)
        f1_test = f1_score(y_test, y_pred_test, average="macro")
        precision_test = precision_score(y_test, y_pred_test, average="macro")

        results.append({
            "Time of Trial (Seconds)": end_time - start_time, 
            "Features Selected": X_train_red.shape[1], 
            "Train Accuracy": acc_train,
            "Train F1 Score": f1_train,
            "Train Precision": precision_train,
            "Test Accuracy": acc_test,
            "Test F1 Score": f1_test,
            "Test Precision": precision_test
        })

    return pd.DataFrame(results).mean()

### Human Activity Recognition

In [12]:
# Load the HAR dataset
X_har, y_har = fetch_openml(
    'har', version=1, return_X_y=True, parser='auto'
)

In [13]:
threfhold_har = eval_method(
    ThreFholdDimensionReducer(), 
    X_har.to_numpy(), y_har.to_numpy()
)

bh_har = eval_method(
    BHDimensionReducer(), 
    X_har.to_numpy(), y_har.to_numpy()
)

l2_har = eval_method(
    L2DimensionReducer(), 
    X_har.to_numpy(), y_har.to_numpy()
)

fpr_har = eval_method(
    FPRDimensionReducer(), 
    X_har.to_numpy(), y_har.to_numpy()
)

har_results = pd.DataFrame({
    "ThreFhold": threfhold_har, 
    "Benjamini-Hochberg": bh_har, 
    "L2": l2_har, 
    "False Positive Rate": fpr_har
}).T

har_results

/var/folders/cn/w9246zrd5kd2ns6x7dyz682m0000gn/T/ipykernel_7600/3248130928.py:24: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b


,Time of Trial (Seconds),Features Selected,Train Accuracy,Train F1 Score,Train Precision,Test Accuracy,Test F1 Score,Test Precision
ThreFhold,0.062910,523.0,1.0,1.0,1.0,0.978833,0.978545,0.978554
Benjamini-Hochberg,0.013677,556.0,1.0,1.0,1.0,0.978153,0.978024,0.978021
L2,0.760044,186.6,1.0,1.0,1.0,0.979610,0.979795,0.979774
False Positive Rate,0.013683,556.0,1.0,1.0,1.0,0.978153,0.978024,0.978021


### Gisette

In [14]:
# Load the Gisette dataset
X_gisette, y_gisette = fetch_openml(
    'gisette', version=2, return_X_y=True, parser='auto'
)

In [15]:
threfhold_gisette = eval_method(
    ThreFholdDimensionReducer(), 
    X_gisette.toarray(), y_gisette
)

bh_gisette = eval_method(
    BHDimensionReducer(), 
    X_gisette.toarray(), y_gisette
)

l2_gisette = eval_method(
    L2DimensionReducer(), 
    X_gisette.toarray(), y_gisette
)

fpr_gisette = eval_method(
    FPRDimensionReducer(), 
    X_gisette.toarray(), y_gisette
)

gisette_results = pd.DataFrame({
    "ThreFhold": threfhold_gisette, 
    "Benjamini-Hochberg": bh_gisette, 
    "L2": l2_gisette, 
    "False Positive Rate": fpr_gisette
}).T

gisette_results

/var/folders/cn/w9246zrd5kd2ns6x7dyz682m0000gn/T/ipykernel_7600/3248130928.py:24: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b
/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [ 119  421  479  539  625  793  876 1004 1102 1639 1676 1681 1792 1825
 1835 2022 2070 2198 2248 2348 2371 2585 2604 2686 2718 2785 2811 2901
 2939 3007 3026 3193 3476 3556 3629 3631 3706 3864 4066 4100 4255 4521
 4634 4650 4964] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [ 119  421  479  539  625  793  876 1004 1102 1639 1676 1681 1792 182

,Time of Trial (Seconds),Features Selected,Train Accuracy,Train F1 Score,Train Precision,Test Accuracy,Test F1 Score,Test Precision
ThreFhold,0.274690,2804.2,1.0,1.0,1.0,0.974714,0.974691,0.974815
Benjamini-Hochberg,0.084081,1868.8,1.0,1.0,1.0,0.975857,0.975839,0.975895
L2,0.281806,1958.8,1.0,1.0,1.0,0.972429,0.972405,0.972483
False Positive Rate,0.096586,2048.4,1.0,1.0,1.0,0.974286,0.974264,0.974349


### Madelon

In [16]:
# Load the Madelon dataset
X_madelon, y_madelon = fetch_openml(
    'madelon', version=1, return_X_y=True, parser='auto'
)

In [17]:
threfhold_madelon = eval_method(
    ThreFholdDimensionReducer(), 
    X_madelon.to_numpy(), y_madelon
)

bh_madelon = eval_method(
    BHDimensionReducer(), 
    X_madelon.to_numpy(), y_madelon
)

l2_madelon = eval_method(
    L2DimensionReducer(), 
    X_madelon.to_numpy(), y_madelon
)

fpr_madelon = eval_method(
    FPRDimensionReducer(), 
    X_madelon.to_numpy(), y_madelon
)

madelon_results = pd.DataFrame({
    "ThreFhold": threfhold_madelon, 
    "Benjamini-Hochberg": bh_madelon, 
    "L2": l2_madelon, 
    "False Positive Rate": fpr_madelon
}).T

madelon_results

/var/folders/cn/w9246zrd5kd2ns6x7dyz682m0000gn/T/ipykernel_7600/3248130928.py:24: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp((k - k_shapiro_w) * (skewness - x0))) + b


,Time of Trial (Seconds),Features Selected,Train Accuracy,Train F1 Score,Train Precision,Test Accuracy,Test F1 Score,Test Precision
ThreFhold,0.013138,66.0,1.0,1.0,1.0,0.781923,0.781500,0.783731
Benjamini-Hochberg,0.003337,14.2,1.0,1.0,1.0,0.864615,0.864505,0.864724
L2,0.019388,175.2,1.0,1.0,1.0,0.768462,0.767894,0.770266
False Positive Rate,0.003266,41.8,1.0,1.0,1.0,0.805000,0.804610,0.806532
